# 12주차 실습 — CDW 분석에서 자주 생기는 오류

CDW 분석 오류는 문법이 아니라 데이터에서 나옵니다.

- **데이터**: `aact_mini` — 7~9주차와 같은 임상시험 데이터입니다.
- **사용법**: 오류마다 동료의 SQL → 점검 SQL → `___` 를 채워 실행하면 결과 아래에 ✅ / ❌ 가 나옵니다.

**오늘 배우는 것**

1. 건수가 부풀거나 줄어드는 원인을 찾아낼 수 있다
2. 타입과 형식 때문에 생기는 오류를 알아볼 수 있다
3. NULL과 표기 불일치를 확인하고 처리할 수 있다
4. 점검 체크리스트로 결과를 스스로 검증할 수 있다

## 0. 시작하기 전에

- Colab 에서는 먼저 **`파일 → Drive에 사본 저장`** 을 누르세요.
- 셀은 **위에서부터 차례로** 실행합니다 (▶ 또는 `Shift + Enter`). `%%sql` 아래의 `-- Q1` 같은 줄은 지우지 마세요.
- 문항 SQL 을 실행하면 결과 아래에 **✅ / ❌** 가 바로 나옵니다.
- 오류가 나면 **`… Error:` 로 시작하는 줄**만 읽으면 됩니다.

In [ ]:
#@title [1번] ▶ 준비 셀 — 실행만 하세요 (Shift+Enter). 코드는 몰라도 됩니다. { display-mode: "form" }
import os, sys, urllib.request
import numpy as _np, pandas as _pd
DB  = "aact_mini.duckdb"
URL = "https://raw.githubusercontent.com/kevinkwshin/medical-db-course-2026/main/data/aact_mini.duckdb.gz"

# 로컬(수업 폴더)에서 열었을 때는 ../data/ 의 파일을 사용
for cand in [DB, os.path.join("..", "data", DB), os.path.join("data", DB)]:
    if os.path.exists(cand):
        DB = cand
        break
else:
    print("데이터 파일을 내려받습니다 (약 50MB, 1분 내외) ...")
    import gzip, shutil
    urllib.request.urlretrieve(URL, DB + ".gz")
    with gzip.open(DB + ".gz", "rb") as f_in, open(DB, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)
    os.remove(DB + ".gz")

if "google.colab" in sys.modules:           # Colab 에서만 설치 (수업에서 검증한 버전)
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb==1.4.5", "jupysql==0.11.1", "duckdb-engine==0.17.0"])

import duckdb
_pd.set_option("display.max_colwidth", 60)
%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.displaylimit = 30
%config SqlMagic.feedback = 0
%xmode Minimal
try:                                         # jupysql 의 영어 안내문(If using snippets ...)을 숨김
    import sql.error_handler as _eh
    _eh.CTE_MSG, _eh.ORIGINAL_ERROR = "↓ 아래 '… Error:' 로 시작하는 줄이 원인입니다", ""
except Exception:
    pass
conn = duckdb.connect(DB, read_only=True)
conn.execute("SET python_enable_replacements = false")   # 빈칸 ___ 을 파이썬 변수로 읽지 않게

class _SameCursor:                           # jupysql 이 셀마다 새 cursor(=새 연결)를 열면
    def __init__(self, c): self._c = c       # CREATE TEMP TABLE 로 만든 표가 다음 셀에서 사라지므로
    def close(self): pass                    # 항상 같은 연결을 쓰게 합니다
    def __getattr__(self, k): return getattr(self._c, k)

class _OneConnection:
    def __init__(self, c): self._c = c
    def cursor(self): return _SameCursor(self._c)
    def __getattr__(self, k): return getattr(self._c, k)

_aact = _OneConnection(conn)
%sql _aact --alias aact

import hashlib as _hl, math as _math, decimal as _dec, datetime as _dt
import re as _re, zlib as _zlib, base64 as _b64, json as _json

def _load_answers(blob):
    """자동 확인용 지문(정답 결과표의 해시 · 행 수 · 열 이름 · 힌트)을 풉니다. 정답 SQL 은 들어 있지 않습니다."""
    return _json.loads(_zlib.decompress(_b64.b64decode(blob)).decode())

def _v(v):
    """값 하나를 비교용 문자열로: 빈 값 → '', 1947.0 → '1947', 날짜 → 'YYYY-MM-DD'"""
    try:
        if v is None or (not isinstance(v, (str, bytes)) and _pd.isna(v)):
            return ""
    except (TypeError, ValueError):
        pass
    if isinstance(v, _dec.Decimal):
        v = float(v)
    if isinstance(v, (bool, _np.bool_)):
        return str(bool(v))
    if isinstance(v, (int, _np.integer)):
        return str(int(v))
    if isinstance(v, (float, _np.floating)):
        v = round(float(v), 4)
        return str(int(v)) if v.is_integer() else repr(v)
    if isinstance(v, (_pd.Timestamp, _dt.datetime)):
        if v.hour == 0 and v.minute == 0 and v.second == 0 and v.microsecond == 0:
            return v.strftime("%Y-%m-%d")
        return v.isoformat()
    if isinstance(v, _dt.date):
        return v.isoformat()
    return str(v)

def _sigs(df):
    """결과표의 지문: 열 이름(소문자), 행 수, 행 순서 무시 지문(sig), 행 순서 포함 지문(osig)"""
    cols = [str(c).strip().lower() for c in df.columns]
    rows = ["\x1f".join(_v(x) for x in r) for r in df.itertuples(index=False, name=None)]
    head = "\x1e".join(cols)
    h = lambda rs: _hl.md5((head + "\x1d" + "\x1e".join(rs)).encode()).hexdigest()[:12]
    return {"sig": h(sorted(rows)), "osig": h(rows), "rows": len(rows), "cols": cols}

_LAST = {"tag": None, "error": False, "sql": ""}
_TRIES, _RESULT = {}, {}

def _after_cell(result):
    info = getattr(result, "info", None)
    src = (getattr(info, "raw_cell", "") or "").lstrip()
    if not src.startswith("%%sql"):
        return
    tag = None
    for line in src.split("\n")[1:]:
        s = line.strip()
        if s.startswith("--"):
            parts = s[2:].split()
            if parts and _re.match(r"^\[\d+번\]$", parts[0]):      # 수업용 셀 번호 [12번] 은 건너뜀
                parts = parts[1:]
            tag = parts[0] if parts and not parts[0].startswith("[") else None
            break
        if s:
            break
    body = "\n".join(l.split("--")[0] for l in src.split("\n")[1:])
    _LAST.update(tag=tag, error=not result.success, sql=body)
    if tag and tag in globals().get("ANSWERS", {}):         # 문항 셀(-- Q1)이면 실행하자마자 ✅/❌ 표시
        check(tag, getattr(result, "result", None))

for _cb in list(get_ipython().events.callbacks["post_run_cell"]):
    if getattr(_cb, "__name__", "") == "_after_cell":
        get_ipython().events.unregister("post_run_cell", _cb)
get_ipython().events.register("post_run_cell", _after_cell)

def _expected(name):
    a = ANSWERS[name]
    if "by_mesh" in a:
        if not globals().get("MY_MESH"):
            return None
        return dict(a["by_mesh"][MY_MESH], hint=a.get("hint", ""), ordered=a.get("ordered", False))
    return a

def _miss(name, why, exp):
    """틀렸을 때: 기록하고, 두 번째부터 힌트를 붙여 출력"""
    _RESULT[name] = False
    _TRIES[name] = _TRIES.get(name, 0) + 1
    msg = f"❌ {name}  {why}"
    hint = (exp or ANSWERS.get(name, {})).get("hint")
    if hint:
        msg += ("\n   💡 힌트: " + hint) if _TRIES[name] >= 2 else "\n   (한 번 더 다르면 힌트가 나옵니다)"
    print(msg)

def check(name, df=None):
    """바로 위 %%sql 셀의 결과를 확인합니다.  예) check("Q1", _)"""
    if name not in ANSWERS:
        print(f"알 수 없는 문항 이름입니다: {name}"); return
    if _LAST["tag"] and _LAST["tag"] != name:
        print(f"⚠️ 방금 실행한 SQL 셀은 {_LAST['tag']} 입니다. {name} 의 %%sql 셀을 먼저 실행한 뒤 이 셀을 실행하세요."); return
    exp = _expected(name)
    if exp is None:
        print("⚠️ 분석 질환 정보가 없습니다. 맨 위 준비 셀을 다시 실행하세요."); return
    if _LAST["sql"] and not _re.search(r"\bFROM\b", _LAST["sql"], _re.I):
        return _miss(name, "FROM 이 없는 SQL 입니다. 숫자를 직접 적지 말고, 표에서 꺼내 세는 SQL 을 쓰세요.", exp)
    if _LAST["error"]:
        return _miss(name, "SQL 에서 오류가 났습니다. 붉은 글씨 중 'Error:' 가 들어간 줄을 읽고 고친 뒤 다시 실행하세요.", exp)
    if not isinstance(df, _pd.DataFrame):
        return _miss(name, "결과가 표가 아닙니다. SQL 이 SELECT 로 끝나는지 확인하세요.", exp)
    got = _sigs(df)
    if got["sig"] == exp["sig"] and (not exp.get("ordered") or got["osig"] == exp["osig"]):
        _RESULT[name] = True
        print(f"✅ {name} 정답!  (행 {got['rows']}개)"); return
    zero = "값의 철자 · 대소문자 · 따옴표(예: 'COMPLETED')와, 빈 값을 = 나 <> 로 비교하지 않았는지(IS NULL) 확인하세요."
    if got["sig"] == exp["sig"]:
        why = "값은 맞지만 행의 순서가 다릅니다. 정렬(ORDER BY · DESC)을 확인하세요."
    elif got["cols"] != exp["cols"]:
        if len(got["cols"]) != len(exp["cols"]):
            why = f"열 개수가 다릅니다 (기대 {len(exp['cols'])}개: {', '.join(exp['cols'])} / 현재 {len(got['cols'])}개)."
        elif sorted(got["cols"]) == sorted(exp["cols"]):
            why = f"열의 순서가 다릅니다. 문항에 적힌 순서대로 적으세요: {', '.join(exp['cols'])}"
        else:
            why = f"열 이름이 다릅니다. 기대: {', '.join(exp['cols'])}  /  현재: {', '.join(got['cols'])}  (AS 로 이름을 맞추세요)"
    elif got["rows"] == 0 and exp["rows"] > 0:
        why = "결과가 0행입니다. " + zero
    elif got["rows"] == 1 and len(df.columns) and all(_v(x) in ("0", "") for x in df.iloc[0]):
        why = "결과가 0 이거나 비어 있습니다. 조건에 맞는 행이 없다는 뜻입니다. " + zero
    elif got["rows"] != exp["rows"]:
        why = f"행 수가 다릅니다 (기대 {exp['rows']}행, 현재 {got['rows']}행). 조건과 LIMIT 을 확인하세요."
    elif exp.get("ordered"):
        why = "행 수는 맞지만 값이나 순서가 다릅니다. 조건과 정렬(ORDER BY · DESC)을 확인하세요."
    else:
        why = "행 수는 맞지만 값이 다릅니다. 조건이나 계산식을 확인하세요."
    _miss(name, why, exp)

def 진행확인():
    """지금까지 확인한 문항의 결과를 한눈에 봅니다. 선택 문항은 개수에서 뺍니다."""
    names = list(ANSWERS)
    core = [n for n in names if not ANSWERS[n].get("optional")]
    ok = [n for n in core if _RESULT.get(n) is True]
    print(f"정답 {len(ok)} / {len(core)} 문항")
    for n in names:
        mark = {True: "✅", False: "❌"}.get(_RESULT.get(n), "·  (아직 실행 안 함)")
        print(f"  {n:>4}  {mark}" + ("   (선택)" if ANSWERS[n].get("optional") else ""))

print("연결 완료:", DB)

ANSWERS = _load_answers("eNqllN9r01AUx/+VQ546GKNJ2rTzTV/mBCdD8UWkpLmpBkYrTYYPY1BmHJmr2LkfzWZTK3Tuhz6Utm6d1Bf/nNyb/8Fzs1bS0qHbIJB7v+eecD7ne09WhEVRuAMrgmm8wLegaXF1VsqKIpGIMA1C4Qq9WHhtoi7iUiss8eUzgW3WgqrLHFd4jvJLI2/xzGBvg3ldYLs23XSo14fLY0DbTuD2IK9ZGYMAPcRAc4t2fgCrrocJdo8/BzszYR1Fohd1gh/MqUumzpVXllHIq0tDaRW1RSnKIitKNpFTVJIm6SjLuP7/LMxt0kMXRNZ26ecJBdOKOw10qwLBVo1VK0BPj/mO1R36bmeIynbrGOMCngLMpF9tMK1lYujmTNgIeraP8vXp5Sh9kuRySU1OJxQxGaUf1yfR0zMbSx9Bf/BofgGLLQE78kLa6jpHGPTJs4H+fM8N3mzOANvvs9NfYRM8F8ILYA97QdeO/V6Lpw6+aAPbbl2fNRFlTeQkScrJmq5ocpR1XJ/EqhWW81bGtNRibGrU7EbFb5fAb7f8Th8dY3YNgjclVn/L+e/Nz80vPIkx5xurV6aY18OOdG9Mk4zSKGpKzKZTybisilGacf1WNEA7XX730Ai/9TE0qFxi62X6vYdA8PscqGezvS7ioF/+RZkdYfZ5D0/RL7WbUSpRSkJScVGdFWfT4sh0jutXUsaIYVoGDtVgsEZ5szxPz2iFPDF4KeZgJrmLD/XH98NtE6/zwQlS+i1cfarQust1fkEvMel2E6PDAb6Nw6kou5iU5SzRsykpFY+yj+sT/0wNm7W7PMp9qzvMK/NN8GHDP2+MtODu07kYljwVji29cP4afdH3Ow3wOzZbawW7J4Ox/SeMVVzWV1f/AHAui4Y=")   # 자동 확인용 (정답 SQL 은 없음)

## 1. 세는 단위와 JOIN

### 오류 1 — 세는 단위를 잘못 잡음

**상황** “한국에서 진행된 임상시험이 몇 건인가요?”

In [ ]:
%%sql
-- [2번] [오류 예시] 동료의 SQL — 실행은 되지만 답이 틀립니다
SELECT COUNT(*) AS 시험수
FROM facilities
WHERE country = 'South Korea';

In [ ]:
%%sql
-- [3번] 점검 ① 한 행의 단위 확인 — 행 수와 시험 수가 같은가요?
SELECT COUNT(*), COUNT(DISTINCT nct_id)
FROM facilities;

**Q1.** 한 시험이 여러 기관에서 진행되어 기관 기록의 **행 수**를 센 것입니다. 시험 단위로 중복을 없애도록 고치세요.

In [ ]:
%%sql
-- [4번] Q1
-- 이렇게 고칩니다 — ___ 를 채우세요
SELECT COUNT(___) AS 시험수
FROM facilities
WHERE country = 'South Korea';

> CDW에서도 같습니다. 환자 수인지, 방문 건수인지, 처방 건수인지 먼저 정하세요. 표의 한 행이 무엇인지부터 확인합니다.

### 오류 2 — JOIN 뒤 행이 늘어남

**상황** 질환 정보를 붙였더니 건수가 10배가 되었습니다.

In [ ]:
%%sql
-- [5번] [오류 예시] 동료의 SQL — 실행은 되지만 답이 틀립니다
SELECT COUNT(*) AS 시험수
FROM studies
JOIN browse_conditions
  ON studies.nct_id = browse_conditions.nct_id;

In [ ]:
%%sql
-- [6번] 점검 ① 한 행의 단위 확인 — 한 시험에 질환이 몇 행인가요?
SELECT COUNT(*), COUNT(DISTINCT nct_id)
FROM browse_conditions;

**Q2.** 한 시험에 질환이 평균 10개 붙어 행이 늘어났습니다. **시험 단위를 유지**하도록 고치세요.

In [ ]:
%%sql
-- [7번] Q2
-- 이렇게 고칩니다 — ___ 를 채우세요
SELECT COUNT(___) AS 시험수
FROM studies
JOIN browse_conditions
  ON studies.nct_id = browse_conditions.nct_id;

> JOIN 전후의 행 수를 항상 비교하세요. 1 : N 이면 반드시 늘어납니다.

### 오류 3 — INNER JOIN으로 분모가 줄어듦

**상황** “전체 시험 중 한국이 참여한 비율”을 구하려고 합니다. 먼저 분모를 셉니다.

In [ ]:
%%sql
-- [8번] [오류 예시] 동료의 SQL — 실행은 되지만 답이 틀립니다
SELECT COUNT(DISTINCT studies.nct_id) AS 분모
FROM studies
JOIN facilities
  ON studies.nct_id = facilities.nct_id;

In [ ]:
%%sql
-- [9번] 점검 분모 — 전체 시험 수와 동료의 분모를 비교하세요
SELECT COUNT(*) AS 전체시험
FROM studies;

**Q3.** 기관 정보가 없는 2,699건이 조용히 빠졌습니다. **전체를 분모로 유지**하도록 JOIN 의 종류를 바꾸세요.

In [ ]:
%%sql
-- [10번] Q3
-- 이렇게 고칩니다 — ___ 를 채우세요
SELECT COUNT(DISTINCT studies.nct_id) AS 분모
FROM studies
___ JOIN facilities
  ON studies.nct_id = facilities.nct_id;

> 비율을 구할 때는 분모가 줄지 않았는지 LEFT JOIN 으로 확인하세요. CDW에서는 검사 · 처방 기록이 없는 환자가 이렇게 빠집니다.

## 2. 값의 타입과 형식

### 오류 4 — 문자로 저장된 숫자

**상황** “최소 연령 60세 이상인 시험”을 찾으려고 합니다.

In [ ]:
%%sql
-- [11번] [오류 예시] 동료의 SQL — 실행은 되지만 답이 틀립니다
SELECT COUNT(*)
FROM eligibilities
WHERE minimum_age >= '60 Years';

In [ ]:
%%sql
-- [12번] 점검 값의 타입 확인 — VARCHAR = 문자, BIGINT = 숫자
SELECT table_name, column_name, data_type
FROM information_schema.columns
WHERE column_name LIKE 'minimum_age%';

**Q4.** 글자 순서로 비교해서 '9 Years' 가 '60 Years' 보다 크게 나옵니다. **숫자형 열을 사용**하도록 고치세요.

In [ ]:
%%sql
-- [13번] Q4
-- 이렇게 고칩니다 — ___ 를 채우세요
SELECT COUNT(*)
FROM calculated_values
WHERE ___ >= 60
  AND minimum_age_unit = 'year';

> CDW에서는: 혈압 `'120/80'`, 검사결과 `'<0.5'` 처럼 기호가 섞인 값은 모두 문자입니다.

## 3. 빈 값과 코드

### 오류 9 — 대소문자와 공백

**상황** 임상 단계를 'Phase 3' 으로 찾았습니다.

In [ ]:
%%sql
-- [14번] [오류 예시] 동료의 SQL — 실행은 되지만 답이 틀립니다
SELECT COUNT(*)
FROM studies
WHERE phase = 'Phase 3';

In [ ]:
%%sql
-- [15번] 점검 표기 불일치 — 값을 먼저 보고 조건 작성
SELECT DISTINCT phase
FROM studies;

**Q5.** 표기가 다르면 하나도 안 나옵니다. 점검 결과에서 3상의 **실제 값**을 찾아 `___` 에 그대로 쓰세요.

In [ ]:
%%sql
-- [16번] Q5
-- 이렇게 고칩니다 — ___ 를 채우세요
SELECT COUNT(*)
FROM studies
WHERE phase = '___';

> 값이 0행이면 “없다”가 아니라 “조건이 틀렸다”를 먼저 의심하세요.

### 오류 10 — 같은 질환, 다른 표기

**상황** “류마티스 관절염” 시험을 연구자 입력 값으로 찾았습니다. (슬라이드: 제2형 당뇨병)

In [ ]:
%%sql
-- [17번] [오류 예시] 동료의 SQL — 실행은 되지만 답이 틀립니다
SELECT COUNT(DISTINCT nct_id)
FROM conditions
WHERE name = 'Arthritis, Rheumatoid';

In [ ]:
%%sql
-- [18번] 점검 ② 값의 종류 확인 — 같은 질환이 몇 가지로 적혀 있나요?
SELECT name, COUNT(*)
FROM conditions
WHERE name ILIKE '%rheumatoid%'
GROUP BY name
ORDER BY 2 DESC;   -- 2 = 두 번째 열

**Q6.** 자유 입력이라 표기가 여러 가지로 갈라져 있습니다. **MeSH 표준 용어 열**을 사용하도록 고치세요.

In [ ]:
%%sql
-- [19번] Q6
-- 이렇게 고칩니다 — ___ 를 채우세요
SELECT COUNT(DISTINCT nct_id)
FROM browse_conditions
WHERE ___ = 'Arthritis, Rheumatoid';

> CDW에서는: 진단명 자유 입력 대신 KCD 코드를 쓰는 이유가 바로 이것입니다.

## 4. 시간

오류 12와 실습하지 않은 오류를 표로 봅니다.

| 오류 | 이렇게 하면 | 이렇게 고칩니다 |
|---|---|---|
| **오류 5** 문자로 저장된 날짜 | `'2024-9-1' < '2024-10-1'` → false | `DATE '2024-09-01' < DATE '2024-10-01'` → true |
| **오류 6** 정수 나눗셈으로 0이 나옴 | `191 / 817 * 100` → 0 (PostgreSQL 등) | `ROUND(100.0 * 191 / 817, 1)` → 23.4 |
| **오류 7** 엑셀에서 앞자리 0이 사라짐 | '0012345' → 12345 로 바뀌어 0행 | CSV를 열 때 그 열을 ‘텍스트’로 지정 |
| **오류 8** NULL을 0으로 바꿈 | `AVG(COALESCE(검사값, 0))` → 실제보다 낮음 | `COUNT(*)` · `COUNT(검사값)` · `AVG(검사값)` 을 함께 봄 |
| **오류 11** 청구 자료를 진료 기록으로 읽음 | 청구(EDI) 자료로 센 유병률이 실제보다 크게 나옴 | 세는 규칙을 문서로 남김 |
| **오류 12** 기준일 이후 정보가 섞임 | 퇴원 후 검사까지 넣음 → 예측 정확도가 비현실적으로 높음 | `WHERE 검사.검사일 <= 입원.입원일` |

## 5. 점검 체크리스트

결과를 내보내기 전에 이 표를 훑습니다.

| 무엇을 | 어떻게 확인하나 | 이상하면 |
|---|---|---|
| 세는 단위 | `COUNT(*)` 와 `COUNT(DISTINCT 키)` 비교 | DISTINCT 를 쓰거나 먼저 집계 |
| JOIN 후 행 수 | JOIN 전후의 행 수 비교 | 1:N 관계 확인, 필요한 단위로 집계 |
| 분모 | LEFT JOIN 으로 다시 세기 | INNER JOIN 을 LEFT JOIN 으로 |
| 값의 타입 | DESCRIBE 로 타입 확인 | 숫자형 열을 쓰거나 CAST |
| 빈 값 | `COUNT(*)` 와 `COUNT(열)` 비교 | NULL 처리 규칙을 정하고 기록 |
| 표기 불일치 | DISTINCT 로 값의 종류 보기 | 표준 코드 열을 사용 |
| 이상값 | MIN · MAX · 상위 10건 보기 | 원인 확인 후 제외 기준 문서화 |
| 기준일 | 기준일 이후 자료가 섞였는지 | 날짜 조건을 다시 확인 |

**오류를 찾는 SQL 네 가지** — 새 데이터를 받으면 가장 먼저 돌려 보는 조회입니다.

```sql
-- ① 한 행의 단위 확인
SELECT COUNT(*), COUNT(DISTINCT 키) FROM 표;
-- ② 값의 종류 확인
SELECT 열, COUNT(*) FROM 표 GROUP BY 열
ORDER BY 2 DESC;   -- 2 = 두 번째 열
-- ③ 빈 값 비율 확인
SELECT COUNT(*) AS 전체, COUNT(열) AS 값있음 FROM 표;
-- ④ 이상값 확인
SELECT MIN(열), MAX(열), AVG(열) FROM 표;
```

이 네 가지를 먼저 돌리면 대부분의 오류를 분석 시작 전에 발견합니다.

### 더 해보기 (선택): 오류 8 — NULL을 0으로 바꿈

**상황** 실제 기간(`actual_duration`)이 빈 시험을 0으로 채워 평균을 냈습니다.

In [ ]:
%%sql
-- [20번] [오류 예시] 동료의 SQL — 실행은 되지만 답이 틀립니다
SELECT AVG(COALESCE(actual_duration, 0)) AS 평균
FROM calculated_values;

**Q7.** 빈 값의 개수를 먼저 확인하고, 평균에서는 제외하도록 `___` 를 채우세요.

In [ ]:
%%sql
-- [21번] Q7
-- 이렇게 고칩니다 — ___ 를 채우세요
SELECT COUNT(*) AS 전체,
       COUNT(actual_duration) AS 값있음,
       AVG(___) AS 평균
FROM calculated_values;

## 오늘 배운 것

- **세는 단위**와 **JOIN 후 행 수**를 항상 확인한다
- **타입 · 표기 · 빈 값**이 결과를 조용히 바꾼다
- **기준일 이전 자료만** 쓰고, 처리 방법을 보고서에 기록한다

과제 2(20점)는 오늘 공개되고, 13주차 수업 후 1주일 이내에 제출합니다.

다음 시간: 13강. CDW 실무 사례 — 코호트 추출을 처음부터 끝까지, 과제 2 제출

In [ ]:
진행확인()   # [22번]